In [5]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import urllib

# ---------------------------------------------------------
# 0. Load Data from CSV Files
# ---------------------------------------------------------
print("Loading data from CSV files...")

calendar_df = pd.read_csv('Calendar.csv', parse_dates=['Date'])
sources_df = pd.read_csv('Sources.csv')
themes_df = pd.read_csv('Themes.csv')
users_df = pd.read_csv('Users.csv')
voc_mentions_df = pd.read_csv('VOC_Mentions.csv', parse_dates=['Date'])
usage_events_df = pd.read_csv('UsageEvents.csv', parse_dates=['Date'])
revenue_events_df = pd.read_csv('RevenueEvents.csv', parse_dates=['Date'])

print("Data loaded successfully.")


# ---------------------------------------------------------
# 1. Clean Calendar Table
# ---------------------------------------------------------
print("Processing Calendar Table...")
calendar_df = calendar_df.drop_duplicates()
calendar_df = calendar_df.sort_values(by='Date')


# ---------------------------------------------------------
# 2. Clean Sources Table
# ---------------------------------------------------------
print("Processing Sources Table...")
sources_df = sources_df.drop_duplicates(subset=['SourceID'])


# ---------------------------------------------------------
# 3. Clean Themes Table
# ---------------------------------------------------------
print("Processing Themes Table...")
themes_df = themes_df.drop_duplicates(subset=['ThemeID'])


# ---------------------------------------------------------
# 4. Clean Users Table
# ---------------------------------------------------------
print("Processing Users Table...")
users_df = users_df.drop_duplicates(subset=['UserID'])
users_df['PMExperienceYears'] = users_df['PMExperienceYears'].clip(lower=0)
users_df['ChurnRiskScore'] = users_df['ChurnRiskScore'].clip(0, 1)


# ---------------------------------------------------------
# 5. Clean VOC_Mentions Table
# ---------------------------------------------------------
print("Processing VOC_Mentions Table...")
voc_mentions_df = voc_mentions_df.drop_duplicates()
voc_mentions_df['SentimentScore'] = voc_mentions_df['SentimentScore'].clip(-1, 1)
voc_mentions_df = voc_mentions_df[voc_mentions_df['UserID'].isin(users_df['UserID'])]
voc_mentions_df = voc_mentions_df[voc_mentions_df['ThemeID'].isin(themes_df['ThemeID'])]
voc_mentions_df = voc_mentions_df[voc_mentions_df['SourceID'].isin(sources_df['SourceID'])]


# ---------------------------------------------------------
# 6. Clean UsageEvents Table
# ---------------------------------------------------------
print("Processing UsageEvents Table...")
usage_events_df = usage_events_df.drop_duplicates()
usage_events_df = usage_events_df[usage_events_df['UserID'].isin(users_df['UserID'])]


# ---------------------------------------------------------
# 7. Clean RevenueEvents Table
# ---------------------------------------------------------
print("Processing RevenueEvents Table...")
# Reset index to avoid the Boolean Series Warning
revenue_events_df = revenue_events_df.drop_duplicates().reset_index(drop=True)

revenue_events_df = revenue_events_df[revenue_events_df['UserID'].isin(users_df['UserID'])]

# Business Logic Check: Remove revenue records that occur AFTER a user has churned
churn_records = usage_events_df[usage_events_df['EventType'] == 'Churn'][['UserID', 'Date']]
churn_records = churn_records.rename(columns={'Date': 'ChurnDate'})

revenue_check = revenue_events_df.merge(churn_records, on='UserID', how='left')

# Fix for Boolean Series Warning: Ensure the mask aligns with the DataFrame
valid_revenue_mask = (revenue_check['ChurnDate'].isna()) | (revenue_check['Date'] <= revenue_check['ChurnDate'])
revenue_events_df = revenue_events_df[valid_revenue_mask]


# ---------------------------------------------------------
# 8. Export to SQL Server
# ---------------------------------------------------------
print("Starting SQL Export...")

SERVER_NAME = 'DESKTOP-931U322\SQLEXPRESS'
DATABASE_NAME = 'ProductVOC_DB'

params = urllib.parse.quote_plus(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={SERVER_NAME};'
    f'DATABASE={DATABASE_NAME};'
    f'Trusted_Connection=yes;'
)

try:
    engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

    # Step A: Clear existing data to prevent duplicates (Facts first, then Dimensions)
    # We use 'if_exists=append', so we must manually clean the tables first.
    # We delete Facts FIRST because they depend on Dimensions (Foreign Key Constraint).
    with engine.connect() as conn:
        print("   Clearing old data from SQL Server...")
        conn.execute(text("DELETE FROM Fact_Revenue_Events"))
        conn.execute(text("DELETE FROM Fact_Usage_Events"))
        conn.execute(text("DELETE FROM Fact_VOC_Mentions"))
        conn.execute(text("DELETE FROM Dim_Users"))
        conn.execute(text("DELETE FROM Dim_Themes"))
        conn.execute(text("DELETE FROM Dim_Sources"))
        conn.execute(text("DELETE FROM Dim_Calendar"))
        conn.commit()

    # Step B: Insert new data (Dimensions first, then Facts)
    # We load Dimensions FIRST so Foreign Keys in Fact tables are valid.
    print("   Exporting Dimensions...")
    calendar_df.to_sql('Dim_Calendar', con=engine, if_exists='append', index=False)
    sources_df.to_sql('Dim_Sources', con=engine, if_exists='append', index=False)
    themes_df.to_sql('Dim_Themes', con=engine, if_exists='append', index=False)
    users_df.to_sql('Dim_Users', con=engine, if_exists='append', index=False)

    print("   Exporting Facts...")
    voc_mentions_df.to_sql('Fact_VOC_Mentions', con=engine, if_exists='append', index=False)
    usage_events_df.to_sql('Fact_Usage_Events', con=engine, if_exists='append', index=False)
    revenue_events_df.to_sql('Fact_Revenue_Events', con=engine, if_exists='append', index=False)

    print("Successfully exported all tables to SQL Server.")

except Exception as e:
    print("Error during SQL export:")
    print(e)

Loading data from CSV files...
Data loaded successfully.
Processing Calendar Table...
Processing Sources Table...
Processing Themes Table...
Processing Users Table...
Processing VOC_Mentions Table...
Processing UsageEvents Table...
Processing RevenueEvents Table...
Starting SQL Export...


C:\Users\w\AppData\Local\Temp\ipykernel_10716\3044159907.py:89: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  revenue_events_df = revenue_events_df[valid_revenue_mask]


   Clearing old data from SQL Server...
   Exporting Dimensions...
   Exporting Facts...
Successfully exported all tables to SQL Server.
